# 🍎 Phi-4-reasoning Model with AIProjectClient 🍏

**Phi-4-reasoning** is a 14B-parameter open-weight reasoning model from Microsoft, fine-tuned from Phi-4 on curated chain-of-thought traces (via supervised fine-tuning) to excel at complex, multi-step reasoning. It punches well above its size on math, coding, and STEM benchmarks — rivaling far larger models — while staying cheap enough to run in many enterprise or personal settings.

In this notebook, you'll see how to:
1. **Initialize** an `AIProjectClient` for your Azure AI Foundry environment.
2. **Chat** with the **Phi-4-reasoning** model using `azure-ai-inference`.
3. **Show** a Health & Fitness example, featuring disclaimers and wellness Q&A.
4. **Enjoy** strong step-by-step reasoning from a small, cost-effective model. 🏋️

> **Disclaimer**: This is not medical advice. Please consult professionals.

## Why Phi-4-reasoning?
A small reasoning model trained specifically to think in explicit steps.
- **Reasoning-First**: Emits `<think>` chain-of-thought before its answer — strong on math, coding, and logic.
- **Small but Capable**: 14B parameters deliver reasoning quality competitive with much larger models, at a fraction of the cost.
- **32K Context Window**: Enough room for long reasoning chains and multi-turn conversations.
- **Open Weight**: Available in the Foundry catalog for flexible deployment.

> **Note**: A stronger sibling, **Phi-4-reasoning-plus**, adds a reinforcement-learning stage for even higher accuracy on hard benchmarks.

<img src="./seq-diagrams/4-phi-4-reasoning.png" width="30%"/>

## 1. Setup

Below, we'll install and import the necessary libraries:
- **azure-ai-projects**: For the `AIProjectClient`.
- **azure-ai-inference**: For calling your model, specifically the chat completions.
- **azure-identity**: For `DefaultAzureCredential`.

Ensure you have a `.env` file with:
```bash
PROJECT_CONNECTION_STRING=<your-conn-string>
MICROSOFT_MODEL=phi-4-reasoning
```

> **Note**: It's recommended to complete the [`3-basic-rag.ipynb`](./3-basic-rag.ipynb) notebook before this one, as it covers important concepts that will be helpful here.

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.inference.models import SystemMessage, UserMessage, AssistantMessage

from pathlib import Path

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent.parent
load_dotenv(parent_dir / '.env')

conn_string = os.getenv("PROJECT_CONNECTION_STRING")
phi4_deployment = os.getenv("MICROSOFT_MODEL", "phi-4-reasoning")

try:
    project_client = AIProjectClient.from_connection_string(
        credential=DefaultAzureCredential(),
        conn_str=conn_string,
    )
    print("✅ AIProjectClient created successfully!")
except Exception as e:
    print("❌ Error creating AIProjectClient:", e)

## 2. Chat with Phi-4-reasoning 🍏
We'll demonstrate a simple conversation using **Phi-4-reasoning** in a health & fitness context. We'll define a system prompt that clarifies the role of the assistant. Then we'll ask some user queries.

> Notice that Phi-4-reasoning is well-suited for chain-of-thought reasoning. We'll let it illustrate its reasoning steps for fun.

In [ ]:
def chat_with_phi4(user_question, chain_of_thought=False):
    """Send a chat request to the Phi-4-reasoning model with optional chain-of-thought."""
    # We'll define a system message with disclaimers:
    system_prompt = (
        "You are a Phi-4-reasoning AI assistant, focusing on health and fitness.\n"
        "Remind users that you are not a medical professional, but can provide general info.\n"
    )

    # We can optionally instruct the model to show chain-of-thought. (Use carefully in production.)
    if chain_of_thought:
        system_prompt += "Please show your step-by-step reasoning in your answer.\n"

    # We create messages for system + user.
    system_msg = SystemMessage(content=system_prompt)
    user_msg = UserMessage(content=user_question)

    with project_client.inference.get_chat_completions_client() as chat_client:
        response = chat_client.complete(
            model=phi4_deployment,
            messages=[system_msg, user_msg],
            temperature=0.8,  # a bit creative
            top_p=0.9,
            max_tokens=400,
        )

    return response.choices[0].message.content

# Example usage:
question = "I'm training for a 5K. Any tips on a weekly workout schedule?"
answer = chat_with_phi4(question, chain_of_thought=True)
print("🗣️ User:", question)
print("🤖 Phi-4-reasoning:", answer)

## 3. RAG-like Example (Stub)
Phi-4-reasoning also excels in retrieval augmented generation scenarios, where you provide external context and let the model reason over it. Below is a **stub** example showing how you'd pass retrieved text as context.

> In a real scenario, you'd embed & search for relevant passages, then feed them into the user/system message.

In [ ]:
def chat_with_phi4_rag(user_question, retrieved_doc):
    """Simulate an RAG flow by appending retrieved context to the system prompt."""
    system_prompt = (
        "You are Phi-4-reasoning, helpful fitness AI.\n"
        "We have some context from the user's knowledge base: \n"
        f"{retrieved_doc}\n"
        "Please use this context to help your answer. If the context doesn't help, say so.\n"
    )

    system_msg = SystemMessage(content=system_prompt)
    user_msg = UserMessage(content=user_question)

    with project_client.inference.get_chat_completions_client() as chat_client:
        response = chat_client.complete(
            model=phi4_deployment,
            messages=[system_msg, user_msg],
            temperature=0.3,
            max_tokens=300,
        )
    return response.choices[0].message.content

# Let's define a dummy doc snippet:
doc_snippet = "Recommended to run 3 times per week and mix with cross-training.\n" \
              "Include rest days or active recovery days for muscle repair."

user_q = "How often should I run weekly to prepare for a 5K?"
rag_answer = chat_with_phi4_rag(user_q, doc_snippet)
print("🗣️ User:", user_q)
print("🤖 Phi-4-reasoning (RAG):", rag_answer)

## 4. Wrap-Up & Best Practices
1. **Chain-of-Thought**: Great for debugging or certain QA tasks, but be mindful about revealing chain-of-thought to end users.
2. **RAG**: Use `azure-ai-inference` with retrieval results to ground your answers.
3. **OpenTelemetry**: Optionally integrate `opentelemetry-sdk` and `azure-core-tracing-opentelemetry` for full observability.
4. **Evaluate**: Use `azure-ai-evaluation` to measure your model’s performance.
5. **Cost & Performance**: Phi-4-reasoning delivers strong reasoning from a small 14B model at low cost. It shines on math/coding/STEM — evaluate for your domain needs.

## 🎉 Congratulations!
You've seen how to:
- Use **Phi-4-reasoning** with `AIProjectClient` and `azure-ai-inference`.
- Create a **chat** flow with chain-of-thought.
- Stub a **RAG** scenario.

> Happy hacking! 🏋️